# Python Exceptions
Covers: try/except/else/finally, exception hierarchy, custom exceptions, raise/raise from, chaining, ExceptionGroup, best practices

## 1. try / except / else / finally

In [ ]:
def safe_divide(a, b):
    """Demonstrates all four clauses of exception handling."""
    try:
        result = a / b
    except ZeroDivisionError:
        print(f'  except: Cannot divide {a} by zero')
        return None
    except TypeError as e:
        print(f'  except: Type error — {e}')
        return None
    else:
        # Only runs if NO exception was raised
        print(f'  else: Success! {a} / {b} = {result}')
        return result
    finally:
        # ALWAYS runs — cleanup
        print(f'  finally: safe_divide({a}, {b}) completed')

print('Case 1: Normal division')
safe_divide(10, 2)

print('\nCase 2: Division by zero')
safe_divide(10, 0)

print('\nCase 3: Type error')
safe_divide(10, 'two')

## 2. Exception Hierarchy

In [ ]:
# Demonstrate exception hierarchy
exceptions_to_test = [
    (lambda: 1/0, 'ZeroDivisionError'),
    (lambda: int('abc'), 'ValueError'),
    (lambda: [][5], 'IndexError'),
    (lambda: {}['missing'], 'KeyError'),
    (lambda: None.attr, 'AttributeError'),
    (lambda: open('nonexistent.txt'), 'FileNotFoundError'),
]

for func, name in exceptions_to_test:
    try:
        func()
    except Exception as e:
        mro = [c.__name__ for c in type(e).__mro__]
        print(f'{name}: {mro}')

# Catching base class catches subclasses
print('\nCatching LookupError catches both IndexError and KeyError:')
for exc_func in [lambda: [][0], lambda: {}['k']]:
    try:
        exc_func()
    except LookupError as e:
        print(f'  Caught {type(e).__name__} via LookupError')

## 3. Custom Exceptions

In [ ]:
# Custom exception hierarchy
class AppError(Exception):
    """Base exception for our application."""
    pass

class ValidationError(AppError):
    """Raised when input validation fails."""
    def __init__(self, field: str, message: str):
        self.field = field
        self.message = message
        super().__init__(f"Validation error on '{field}': {message}")

class NotFoundError(AppError):
    """Raised when a resource is not found."""
    def __init__(self, resource: str, identifier):
        self.resource = resource
        self.identifier = identifier
        super().__init__(f"{resource} with id={identifier!r} not found")

class DatabaseError(AppError):
    """Raised when database operations fail."""
    def __init__(self, operation: str, detail: str = ''):
        self.operation = operation
        super().__init__(f"Database error during {operation}: {detail}")

# Usage
def get_user(user_id: int):
    if not isinstance(user_id, int):
        raise ValidationError('user_id', 'must be an integer')
    if user_id <= 0:
        raise ValidationError('user_id', 'must be positive')
    users = {1: 'Alice', 2: 'Bob'}
    if user_id not in users:
        raise NotFoundError('User', user_id)
    return users[user_id]

test_cases = [1, 99, -1, 'abc']
for uid in test_cases:
    try:
        user = get_user(uid)
        print(f'get_user({uid!r}) = {user}')
    except NotFoundError as e:
        print(f'NotFoundError: {e.resource} #{e.identifier}')
    except ValidationError as e:
        print(f'ValidationError: field={e.field}, msg={e.message}')
    except AppError as e:
        print(f'AppError: {e}')

## 4. raise, raise from, raise from None

In [ ]:
import traceback

# raise from — explicit exception chaining
def load_config(path):
    import json
    try:
        with open(path) as f:
            return json.load(f)
    except FileNotFoundError as e:
        raise RuntimeError(f'Config not found: {path}') from e
    except json.JSONDecodeError as e:
        raise ValueError(f'Invalid JSON in config: {path}') from e

try:
    load_config('/nonexistent/config.json')
except RuntimeError as e:
    print('Caught RuntimeError:', e)
    print('Caused by:', e.__cause__)
    print('Suppress context:', e.__suppress_context__)

# raise from None — suppress chaining
def get_value(d, key):
    try:
        return d[key]
    except KeyError:
        raise KeyError(f"Required key '{key}' is missing") from None

try:
    get_value({'a': 1}, 'b')
except KeyError as e:
    print('\nKeyError (from None):', e)
    print('Cause:', e.__cause__)     # None
    print('Context:', e.__context__) # still set but suppressed

# bare raise — re-raise with original traceback
def process_with_logging(func):
    try:
        return func()
    except Exception as e:
        print(f'Logging error: {type(e).__name__}: {e}')
        raise  # re-raise — preserves original traceback

try:
    process_with_logging(lambda: 1/0)
except ZeroDivisionError:
    print('\nZeroDivisionError re-raised successfully')

## 5. ExceptionGroup (Python 3.11+)

In [ ]:
import sys

if sys.version_info >= (3, 11):
    # ExceptionGroup — multiple exceptions at once
    def validate_form(data):
        errors = []
        if not data.get('name'):
            errors.append(ValueError('name is required'))
        if not data.get('email'):
            errors.append(ValueError('email is required'))
        if data.get('age', 1) < 0:
            errors.append(ValueError('age must be non-negative'))
        if errors:
            raise ExceptionGroup('Form validation failed', errors)
    
    # except* handles ExceptionGroups
    try:
        validate_form({'age': -5})
    except* ValueError as eg:
        print(f'Caught {len(eg.exceptions)} ValueError(s):')
        for exc in eg.exceptions:
            print(f'  - {exc}')
    
    # Mixed exception types
    try:
        raise ExceptionGroup('mixed errors', [
            ValueError('bad value'),
            TypeError('bad type'),
            ValueError('another bad value'),
        ])
    except* ValueError as eg:
        print(f'\nValueErrors: {[str(e) for e in eg.exceptions]}')
    except* TypeError as eg:
        print(f'TypeErrors: {[str(e) for e in eg.exceptions]}')
else:
    print(f'ExceptionGroup requires Python 3.11+, you have {sys.version[:6]}')
    
    # Pre-3.11 approach: collect errors
    def validate_form_old(data):
        errors = []
        if not data.get('name'):
            errors.append('name is required')
        if not data.get('email'):
            errors.append('email is required')
        if errors:
            raise ValueError('; '.join(errors))
    
    try:
        validate_form_old({'age': 25})
    except ValueError as e:
        print('Validation errors:', e)

## 6. Best Practices

In [ ]:
# Best practice 1: Be specific
def parse_int_good(s):
    try:
        return int(s)
    except ValueError:
        return None  # specific, intentional

# Best practice 2: Use .get() instead of try/except for dict access
d = {'key': 'value'}
# BAD: try: v = d['key'] except KeyError: v = 'default'
v = d.get('key', 'default')  # GOOD

# Best practice 3: EAFP vs LBYL
# LBYL (Look Before You Leap) — check first
import os
def read_file_lbyl(path):
    if os.path.exists(path):  # race condition possible!
        with open(path) as f:
            return f.read()
    return None

# EAFP (Easier to Ask Forgiveness than Permission) — Python style
def read_file_eafp(path):
    try:
        with open(path) as f:
            return f.read()
    except FileNotFoundError:
        return None

# Best practice 4: Include context in messages
def process_record(record_id, data):
    if not isinstance(data, dict):
        raise TypeError(
            f'Expected dict for record {record_id}, got {type(data).__name__}: {data!r}'
        )

# Best practice 5: finally for cleanup
def acquire_resource():
    return {'name': 'resource', 'open': True}

def release_resource(r):
    r['open'] = False
    print(f'Released {r["name"]}')

resource = acquire_resource()
try:
    # Use resource
    result = 'processed'
    # raise RuntimeError('simulated error')  # uncomment to test
finally:
    release_resource(resource)  # always runs

print('\nAll best practices demonstrated')
print('parse_int_good("42"):', parse_int_good('42'))
print('parse_int_good("abc"):', parse_int_good('abc'))